# Music Information Retrieval System

A Retrieval-Augmented Generation (RAG) system built with LangChain for intelligent music data analysis and question answering.

## Architecture

1. **Data Ingestion**: Multi-source music data extraction (Spotify API, Genius API, Audio transcription)
2. **Document Processing**: Text segmentation using RecursiveCharacterTextSplitter
3. **Vector Embeddings**: Semantic representations via Google Generative AI embeddings
4. **Vector Storage**: FAISS for efficient similarity search
5. **Retrieval**: Context-aware document retrieval
6. **Generation**: LLM-powered response synthesis

## Components

- **Text Splitters**: Optimized chunking for music content
- **Embeddings**: GoogleGenerativeAIEmbeddings for semantic understanding  
- **Vector Stores**: FAISS integration for scalable search
- **LLMs**: Google Gemini for natural language generation
- **Prompt Engineering**: Structured templates for consistent outputs
- **Multi-modal Support**: Text and audio input processing

## Setup & Dependencies

Install required packages and configure API credentials.

In [ ]:
%pip install --upgrade spotipy lyricsgenius python-dotenv langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu requests openai-whisper pydub

In [ ]:
import os
import spotipy
import lyricsgenius
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyClientCredentials
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

In [ ]:
# Verify API keys
google_api_key = os.getenv("GOOGLE_API_KEY")
spotify_client_id = os.getenv("SPOTIFY_CLIENT_ID")
spotify_client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
genius_token = os.getenv("GENIUS_ACCESS_TOKEN")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is required")

## MusicRAGSystem Class

In [ ]:
class MusicRAGSystem:
    def __init__(self):
        self.sp = self._init_spotify()
        self.genius = self._init_genius()
        self.llm = GoogleGenerativeAI(
            model="gemini-1.5-flash",
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=0.3
        )
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/embedding-001",
            google_api_key=os.getenv("GOOGLE_API_KEY")
        )

    def _init_spotify(self):
        client_id = os.getenv("SPOTIFY_CLIENT_ID")
        client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
        if client_id and client_secret:
            credentials = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
            return spotipy.Spotify(client_credentials_manager=credentials)
        return None

    def _init_genius(self):
        token = os.getenv("GENIUS_ACCESS_TOKEN")
        if token:
            return lyricsgenius.Genius(token)
        return None

# Data Retrieval Methods

In [ ]:
def get_music_data(self, music_id, limit=10):
        results = []
        
        if not self.sp:
            return results
            
        try:
            # Try as artist ID first
            try:
                artist = self.sp.artist(music_id)
                albums = self.sp.artist_albums(music_id, album_type='album', limit=limit)
                for album in albums['items']:
                    album_data = {
                        'type': 'album',
                        'name': album['name'],
                        'artist': artist['name'],
                        'year': album['release_date'][:4],
                        'spotify_url': album['external_urls']['spotify']
                    }
                    results.append(album_data)
                    
                # Get lyrics for artist's popular songs
                if self.genius:
                    top_tracks = self.sp.artist_top_tracks(music_id)
                    for track in top_tracks['tracks'][:3]:
                        try:
                            song = self.genius.search_song(track['name'], artist['name'])
                            if song:
                                song_data = {
                                    'type': 'song',
                                    'title': song.title,
                                    'artist': song.artist,
                                    'lyrics_snippet': song.lyrics[:800] + '...' if len(song.lyrics) > 800 else song.lyrics,
                                    'genius_url': song.url
                                }
                                results.append(song_data)
                        except:
                            continue
                            
            except:
                # Try as playlist ID
                try:
                    playlist = self.sp.playlist(music_id)
                    tracks = self.sp.playlist_tracks(music_id, limit=limit)
                    for item in tracks['items']:
                        if item['track']:
                            track = item['track']
                            track_data = {
                                'type': 'track',
                                'name': track['name'],
                                'artist': track['artists'][0]['name'],
                                'album': track['album']['name'],
                                'spotify_url': track['external_urls']['spotify']
                            }
                            results.append(track_data)
                        
                except:
                    # Try as album ID
                    try:
                        album = self.sp.album(music_id)
                        album_data = {
                            'type': 'album',
                            'name': album['name'],
                            'artist': album['artists'][0]['name'],
                            'year': album['release_date'][:4],
                            'spotify_url': album['external_urls']['spotify']
                        }
                        results.append(album_data)
                        
                        tracks = self.sp.album_tracks(music_id)
                        for track in tracks['items']:
                            track_data = {
                                'type': 'track',
                                'name': track['name'],
                                'artist': track['artists'][0]['name'],
                                'album': album['name'],
                                'spotify_url': track['external_urls']['spotify']
                            }
                            results.append(track_data)
                    except:
                        pass
                        
        except Exception as e:
            pass
            
        return results

# Document Processing

In [ ]:
def create_documents(self, music_data):
        documents = []
        
        for item in music_data:
            if item['type'] == 'album':
                content = f"Album: {item['name']} by {item['artist']} ({item['year']}). Spotify: {item['spotify_url']}"
            elif item['type'] == 'song':
                content = f"Song: {item['title']} by {item['artist']}. Lyrics: {item['lyrics_snippet']}. Genius: {item['genius_url']}"
            elif item['type'] == 'track':
                content = f"Track: {item['name']} by {item['artist']} from {item['album']}. Spotify: {item['spotify_url']}"
            else:
                continue
                
            documents.append(Document(page_content=content, metadata=item))
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        split_docs = text_splitter.split_documents(documents)
        
        return split_docs

# Vector Store Setup

In [ ]:
def build_rag_system(self, documents):
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/embedding-001",
            google_api_key=os.getenv("GOOGLE_API_KEY")
        )
        vectorstore = FAISS.from_documents(documents, embeddings)
        
        llm = ChatGoogleGenerativeAI(
            model="gemini-pro", 
            temperature=0.7,
            google_api_key=os.getenv("GOOGLE_API_KEY")
        )
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a music assistant. Use the provided context to answer questions about music, albums, artists, and lyrics. Be conversational and informative."),
            ("human", "Context: {context}\n\nQuestion: {question}")
        ])
        
        rag_chain = (
            {"context": vectorstore.as_retriever(), "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )
        
        self.vectorstore = vectorstore
        self.rag_chain = rag_chain
        
        return rag_chain

# Query Processing

In [ ]:
def ask_question(self, question):
        if not hasattr(self, 'rag_chain'):
            return "Please build the RAG system first using build_rag_system()"
        
        response = self.rag_chain.invoke(question)
        return response

# Usage Example

In [ ]:
# Initialize the system
rag_system = MusicRAGSystem()

# Get music data using the Music ID
music_data = rag_system.get_music_data(MUSIC_ID)

# Create documents
documents = rag_system.create_documents(music_data)

# Build RAG system
rag_chain = rag_system.build_rag_system(documents)

# Ask questions
response = rag_system.ask_question(USER_QUESTION)
print(response)

#  User Input Section 

In [ ]:
# Configuration - Interactive Input
MUSIC_ID = input("Spotify ID (Artist/Playlist/Album): ").strip()

if not MUSIC_ID:
    MUSIC_ID = "4Z8W4fKeB5YxbusRsdQVPb"  # Default to Radiohead

USER_QUESTION = input("Your question: ").strip()

if not USER_QUESTION:
    USER_QUESTION = "What are the main themes and musical characteristics?"


# Run the Complete System

In [ ]:
# Run the complete RAG pipeline
rag_system = MusicRAGSystem()
music_data = rag_system.get_music_data(MUSIC_ID)
documents = rag_system.create_documents(music_data)
rag_chain = rag_system.build_rag_system(documents)
response = rag_system.ask_question(USER_QUESTION)
print(response)